In [2]:
!pip install wandb -q

In [3]:
import os
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from PIL import Image
import wandb

In [4]:
config = {
    "data": {
        "data_dir": "/kaggle/input/datasets/msambare/fer2013",
        "num_classes": 7,
        "image_size": 48,
        "emotions": ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"],
        "mean": 0.5077,
        "std": 0.2551
    },
    "training": {
        "batch_size": 64,
        "epochs": 50,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "early_stopping_patience": 10,
        "device": "cuda"
    },
    "augmentation": {
        "random_rotation": 10
    },
    "wandb": {
        "project": "facial-emotion-detection",
        "entity": None
    },
    "model": {
        "architecture": "cnn_scratch",
        "dropout": 0.5
    },
    "paths": {
        "checkpoint_dir": "/kaggle/working/"
    }
}

In [5]:
class FERDataset(Dataset):
    def __init__(self, data_dir, split="train", transform=None):
        self.data_dir = os.path.join(data_dir, split)
        self.transform = transform
        self.emotions = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
        self.label_map = {emotion: idx for idx, emotion in enumerate(self.emotions)}
        
        self.image_paths = []
        self.labels = []
        
        for emotion in self.emotions:
            folder = os.path.join(self.data_dir, emotion)
            for img_file in os.listdir(folder):
                self.image_paths.append(os.path.join(folder, img_file))
                self.labels.append(self.label_map[emotion])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_class_weights(self):
        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(self.labels),
            y=self.labels
        )
        return torch.tensor(weights, dtype=torch.float)


def get_transforms(config, split="train"):
    mean = config["data"]["mean"]
    std = config["data"]["std"]
    size = config["data"]["image_size"]

    if split == "train":
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(config["augmentation"]["random_rotation"]),
            transforms.ToTensor(),
            transforms.Normalize(mean=[mean], std=[std])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[mean], std=[std])
        ])


def get_dataloaders(config):
    data_dir = config["data"]["data_dir"]
    batch_size = config["training"]["batch_size"]

    train_dataset = FERDataset(data_dir, split="train", transform=get_transforms(config, "train"))
    test_dataset = FERDataset(data_dir, split="test", transform=get_transforms(config, "test"))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, train_dataset.get_class_weights()

In [6]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes: int = 7, dropout: float = 0.5):
        super(EmotionCNN, self).__init__()

        # Block 1 — basic edges (1→32)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 48→24
            nn.Dropout2d(0.25),
        )

        # Block 2 — facial parts (32→64)
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 24→12
            nn.Dropout2d(0.25),
        )

        # Block 3 — emotion patterns (64→128)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 12→6
            nn.Dropout2d(0.25),
        )

        # Block 4 — high-level combinations (128→256)
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # 6→3
            nn.Dropout2d(0.25),
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x


def build_model(num_classes: int = 7, dropout: float = 0.5) -> EmotionCNN:
    """
    Factory function — called by train.py and evaluate.py.
    Keeps instantiation consistent across the whole project.
    """
    return EmotionCNN(num_classes=num_classes, dropout=dropout)

In [7]:
def setup_wandb(config):
    wandb.init(
        project=config["wandb"]["project"],
        entity=config["wandb"]["entity"],
        config={
            "learning_rate": config["training"]["learning_rate"],
            "batch_size": config["training"]["batch_size"],
            "epochs": config["training"]["epochs"],
            "architecture": config["model"]["architecture"],
            "dropout": config["model"]["dropout"],
        }
    )


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total

In [8]:
# setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

train_loader, test_loader, class_weights = get_dataloaders(config)
class_weights = class_weights.to(device)

model = EmotionCNN(
    num_classes=config["data"]["num_classes"],
    dropout=config["model"]["dropout"]
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=config["training"]["learning_rate"], weight_decay=config["training"]["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=config["training"]["epochs"])

# wandb login
wandb.login()
setup_wandb(config)

# training loop
best_val_acc = 0.0
patience_counter = 0
checkpoint_dir = config["paths"]["checkpoint_dir"]
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(config["training"]["epochs"]):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": scheduler.get_last_lr()[0]
    })

    print(f"Epoch {epoch+1}/{config['training']['epochs']} — train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, "best_model.pt"))
        print(f"  Checkpoint saved — best val_acc: {best_val_acc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= config["training"]["early_stopping_patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break

wandb.finish()
print(f"Training complete. Best val_acc: {best_val_acc:.4f}")

Training on: cuda


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rhazouani062004a (rhazouani062004a-ensam-mekn-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/50 — train_loss: 1.9820 train_acc: 0.1542 val_loss: 1.9278 val_acc: 0.1932
  Checkpoint saved — best val_acc: 0.1932
Epoch 2/50 — train_loss: 1.9360 train_acc: 0.1691 val_loss: 1.8667 val_acc: 0.1549
Epoch 3/50 — train_loss: 1.9282 train_acc: 0.1859 val_loss: 1.8686 val_acc: 0.2159
  Checkpoint saved — best val_acc: 0.2159
Epoch 4/50 — train_loss: 1.8835 train_acc: 0.2155 val_loss: 1.8016 val_acc: 0.2402
  Checkpoint saved — best val_acc: 0.2402
Epoch 5/50 — train_loss: 1.8655 train_acc: 0.2248 val_loss: 1.8393 val_acc: 0.1737
Epoch 6/50 — train_loss: 1.7839 train_acc: 0.2683 val_loss: 1.6471 val_acc: 0.3810
  Checkpoint saved — best val_acc: 0.3810
Epoch 7/50 — train_loss: 1.6740 train_acc: 0.3621 val_loss: 1.5035 val_acc: 0.4237
  Checkpoint saved — best val_acc: 0.4237
Epoch 8/50 — train_loss: 1.5867 train_acc: 0.3978 val_loss: 1.3794 val_acc: 0.4709
  Checkpoint saved — best val_acc: 0.4709
Epoch 9/50 — train_loss: 1.5207 train_acc: 0.4287 val_loss: 1.3293 val_acc: 0.4955
 

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
lr,███████▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
train_acc,▁▁▁▂▂▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████
train_loss,███▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_acc,▂▁▂▂▁▅▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇█▇███████████████
val_loss,███▇▇▅▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
lr,0
train_acc,0.64837
train_loss,0.86116
val_acc,0.63012


Training complete. Best val_acc: 0.6301
